# TourismGPT — Phi-3 Mini Fine-tuning
Run on Google Colab with T4 GPU. Execute cells top to bottom.

In [1]:
# CELL 1 — Install dependencies (restart runtime if prompted)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install datasets huggingface_hub

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-ndddaon7/unsloth_7440aec0c44d46fea8c120ca960283d3
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-ndddaon7/unsloth_7440aec0c44d46fea8c120ca960283d3
  Resolved https://github.com/unslothai/unsloth.git to commit fbe4d08ad09a7635d79b483c1eb81dc5140434b0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 125.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 130.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 17.9 MB/s eta 0:00:00
  

In [2]:
# CELL 2 — Mount Google Drive and copy dataset
from google.colab import drive
drive.mount('/content/drive')

# Option A: copy from Drive (if you uploaded tourism_finetune.jsonl to Drive)
# !cp "/content/drive/MyDrive/tourism_finetune.jsonl" /content/tourism_finetune.jsonl

# Option B: upload directly via Colab Files panel (left sidebar → upload icon)
# then set JSONL_PATH = "/content/tourism_finetune.jsonl" in Cell 3

Mounted at /content/drive


In [4]:
# CELL 3 — Imports and config
import json
import os
import torch
from pathlib import Path
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import FastLanguageModel

# ── Paths ──────────────────────────────────────────────────────────────────
JSONL_PATH   = "/content/drive/MyDrive/Colab Notebooks/tourism_finetune.jsonl"
OUTPUT_DIR   = "/content/tourism_gpt_adapter"
DRIVE_SAVE   = "/content/drive/MyDrive/tourism_gpt_adapter"

# ── Model config ───────────────────────────────────────────────────────────
MODEL_NAME   = "unsloth/Phi-3-mini-4k-instruct"
MAX_SEQ_LEN  = 2048
DTYPE        = None
LOAD_IN_4BIT = True

# ── LoRA config ────────────────────────────────────────────────────────────
LORA_RANK    = 16
LORA_ALPHA   = 16
LORA_DROPOUT = 0

# ── Training config ────────────────────────────────────────────────────────
BATCH_SIZE   = 2
GRAD_ACCUM   = 4
WARMUP_STEPS = 5
MAX_STEPS    = 300
LEARNING_RATE = 2e-4
LR_SCHEDULER = "cosine"
WEIGHT_DECAY = 0.01
SAVE_STEPS   = 100
LOGGING_STEPS = 25
FP16         = not torch.cuda.is_bf16_supported()
BF16         = torch.cuda.is_bf16_supported()
SEED         = 42

print("Config ready ✓")

Config ready ✓


In [5]:
# CELL 4 — Load Phi-3 Mini with Unsloth (4-bit QLoRA)
print("Loading Phi-3 Mini ...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = DTYPE,
    load_in_4bit   = LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_RANK,
    target_modules             = ["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj"],
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = SEED,
    use_rslora                 = False,
    loftq_config               = None,
)

print(model.print_trainable_parameters())

Loading Phi-3 Mini ...
==((====))==  Unsloth 2026.6.7: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/194 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/458 [00:00<?, ?B/s]

Unsloth 2026.6.7 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


trainable params: 29,884,416 || all params: 3,850,963,968 || trainable%: 0.7760
None


In [6]:
# CELL 5 — Load and format dataset
EOS = tokenizer.eos_token

def format_example(row):
    instruction = row["instruction"].strip()
    output      = row["output"].strip()
    return (
        f"<|user|>\n{instruction}<|end|>\n"
        f"<|assistant|>\n{output}<|end|>\n"
        f"{EOS}"
    )

def load_jsonl(path):
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    print(f"Loaded {len(records)} examples")
    texts = [format_example(r) for r in records]
    return Dataset.from_dict({"text": texts})

dataset = load_jsonl(JSONL_PATH)

print("\n── Sample training text ──────────────────────────────")
print(dataset["text"][0][:400])
print("──────────────────────────────────────────────────────")

Loaded 2220 examples

── Sample training text ──────────────────────────────
<|user|>
where can I check if there are any news on the rebate?<|end|>
<|assistant|>
I've observed that you're eager to stay updated on any news about your rebate. To check the latest updates, you can visit our website's "My Account" section. Log in using your credentials, and navigate to the "Rebate Status" or "Refund Status" page. This page will provide you with real-time information on the prog
──────────────────────────────────────────────────────


In [8]:
# CELL 6x — Train
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    args               = SFTConfig(
        dataset_text_field          = "text",
        max_seq_length              = MAX_SEQ_LEN,
        dataset_num_proc            = 2,
        packing                     = False,
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_steps                = WARMUP_STEPS,
        max_steps                   = MAX_STEPS,
        learning_rate               = LEARNING_RATE,
        fp16                        = FP16,
        bf16                        = BF16,
        logging_steps               = LOGGING_STEPS,
        optim                       = "adamw_8bit",
        weight_decay                = WEIGHT_DECAY,
        lr_scheduler_type           = LR_SCHEDULER,
        seed                        = SEED,
        output_dir                  = OUTPUT_DIR,
        save_steps                  = SAVE_STEPS,
        save_total_limit            = 2,
        report_to                   = "none",
    ),
)

gpu_stats = torch.cuda.get_device_properties(0)
max_memory = round(gpu_stats.total_memory / 1024**3, 3)
print(f"GPU: {gpu_stats.name}  |  {max_memory} GB total")
print("\nStarting training ...")

trainer_stats = trainer.train()

print("\n── Training complete ──")
print(f"  Runtime  : {trainer_stats.metrics['train_runtime']:.0f} s")
print(f"  Samples/s: {trainer_stats.metrics['train_samples_per_second']:.1f}")
used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
print(f"  Peak VRAM: {used_memory} GB / {max_memory} GB")

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2220 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
GPU: Tesla T4  |  14.563 GB total

Starting training ...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,220 | Num Epochs = 2 | Total steps = 300
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,884,416 of 3,850,963,968 (0.78% trained)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
25,0.609782
50,0.585338
75,0.602886
100,0.553889
125,0.604056
150,0.623644
175,0.494266
200,0.581636
225,0.588188
250,0.558918


Unsloth: Restored added_tokens_decoder metadata in /content/tourism_gpt_adapter/checkpoint-200/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/tourism_gpt_adapter/checkpoint-200.
Unsloth: Restored added_tokens_decoder metadata in /content/tourism_gpt_adapter/checkpoint-300/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/tourism_gpt_adapter/checkpoint-300.



── Training complete ──
  Runtime  : 1113 s
  Samples/s: 2.2
  Peak VRAM: 5.625 GB / 14.563 GB


In [9]:
# CELL 7 — Save LoRA adapter
print(f"Saving adapter to {OUTPUT_DIR} ...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Adapter saved ✓")

# Copy to Google Drive so it survives after the session ends
import shutil
shutil.copytree(OUTPUT_DIR, DRIVE_SAVE, dirs_exist_ok=True)
print(f"Backed up to Drive: {DRIVE_SAVE} ✓")

Saving adapter to /content/tourism_gpt_adapter ...


Unsloth: Restored added_tokens_decoder metadata in /content/tourism_gpt_adapter/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/tourism_gpt_adapter.


Adapter saved ✓
Backed up to Drive: /content/drive/MyDrive/tourism_gpt_adapter ✓


In [10]:
# CELL 8 — Quick inference test
def ask(question, max_new_tokens=300):
    FastLanguageModel.for_inference(model)
    prompt = f"<|user|>\n{question}<|end|>\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            temperature    = 0.7,
            top_p          = 0.9,
            do_sample      = True,
            pad_token_id   = tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

test_questions = [
    "Plan a 5-day itinerary for Tokyo for a solo budget traveller.",
    "Compare Bali vs Thailand for a honeymoon trip.",
    "What is the estimated budget for a week in Paris?",
    "How do I book a hotel with free cancellation on Booking.com?",
    "My flight was cancelled — how do I claim a refund?",
    "What cultural customs should I know before visiting Japan?",
]

print("── Inference tests ───────────────────────────────────")
for q in test_questions:
    print(f"\nQ: {q}")
    print(f"A: {ask(q)}")
    print("─" * 60)

Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


── Inference tests ───────────────────────────────────

Q: Plan a 5-day itinerary for Tokyo for a solo budget traveller.


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:254: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

A: Here is a 5-day itinerary for Tokyo tailored for a solo budget traveller:

Day 1–2: Arrive and explore the city center. Visit the main landmarks and enjoy local street food.

Day 3–3: Head to nearby attractions. Book a half-day guided tour for deeper cultural immersion.

Day 4: Day trip to a nearby natural or historical site. Wind down and prepare for departure.

Day 5: Depart. Allow at least 3 hours before your flight for airport transfer and check-in.

Tips for solo budget travellers: Book accommodations in advance, carry local currency, and always have a translation app handy.
────────────────────────────────────────────────────────────

Q: Compare Bali vs Thailand for a honeymoon trip.


Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

A: For your query 'What is the estimated budget for a week in Paris?': Budget travellers can expect to spend $50–80/day covering accommodation, meals, and transport. Mid-range budgets of $100–150/day allow more comfort. Always set aside 10–15% for unexpected expenses.
────────────────────────────────────────────────────────────

Q: How do I book a hotel with free cancellation on Booking.com?


Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Here is a suggested itinerary based on your query: How do I book a hotel with free cancellation on Booking.com? Day 1: Arrive and explore the city center. Day 2: Visit major landmarks. Day 3: Day trip to nearby attractions. Adjust based on your pace and interests.
────────────────────────────────────────────────────────────

Q: My flight was cancelled — how do I claim a refund?


Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Here is a suggested itinerary based on your query: My flight was cancelled — how do I claim a refund? Day 1: Arrive and explore the city center. Day 2: Visit major landmarks. Day 3: Day trip to nearby attractions. Adjust based on your pace and interests.
────────────────────────────────────────────────────────────

Q: What cultural customs should I know before visiting Japan?
A: Here are the key cultural customs to know before visiting Japan:

• Remove shoes before entering homes and many restaurants
• Do not tip — it can be considered rude
• Bow slightly when greeting — depth indicates respect level
• Avoid eating or drinking while walking
• Use two hands when giving or receiving business cards or gifts

Showing basic awareness of local customs goes a long way. Locals genuinely appreciate when visitors make the effort.
────────────────────────────────────────────────────────────
